CNNs ko train karte time ek problem yeh hoti hai ki humein bahut saara labeled data chahiye hota hai. Image classification mein humein images ko alag-alag classes mein divide karna padta hai, jo ek manual process hai.

Lekin, hum CNN feature extractors ko train karne ke liye raw (unlabeled) data use karna chah sakte hain, jise **self-supervised learning** kehte hain. Labels ki jagah, hum training images ko network ke input aur output, dono ki tarah use karenge. **Autoencoder** ka main idea yeh hai ki hamare paas ek **encoder network** hoga, jo input image ko kisi **latent space** (aam taur par yeh ek chhote size ka vector hota hai) mein convert karega, aur phir ek **decoder network** hoga, jiska goal original image ko reconstruct karna hoga.

Kyunki hum autoencoder ko original image se zyada se zyada information capture karne ke liye train kar rahe hain taaki accurate reconstruction ho sake, network input image ka sabse accha **embedding** dhoondhne ki koshish karta hai taaki uska meaning samajh sake.

![Autoencoder diagram](../../../../../lessons/4-ComputerVision/09-Autoencoders/images/autoencoder_schema.jpg)

*Image [Keras blog](https://blog.keras.io/building-autoencoders-in-keras.html) se*

Neeche diye gaye zyadatar examples [is article](https://blog.keras.io/building-autoencoders-in-keras.html) se inspired hain.

Chalo MNIST ke liye sabse simple autoencoder banate hain:

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt

(x_train, y_trainclass), (x_test, y_testclass) = mnist.load_data()

In [ ]:
def plotn(n,x):
  fig,ax = plt.subplots(1,n)
  for i,z in enumerate(x[0:n]):
    ax[i].imshow(z.reshape(28,28) if z.size==28*28 else z.reshape(14,14) if z.size==14*14 else z)
  plt.show()
  
plotn(5,x_train)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, UpSampling2D, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.losses import binary_crossentropy,mse

input_img = Input(shape=(28, 28, 1))

x = Conv2D(16, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(8, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(8, (3, 3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x)

encoder = Model(input_img,encoded)

input_rep = Input(shape=(4,4,8))

x = Conv2D(8, (3, 3), activation='relu', padding='same')(input_rep)
x = UpSampling2D((2, 2))(x)
x = Conv2D(8, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2D(16, (3, 3), activation='relu')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

decoder = Model(input_rep,decoded)

autoencoder = Model(input_img, decoder(encoder(input_img)))
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

In [ ]:
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.
x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))

In [ ]:
autoencoder.fit(x_train, x_train,
                epochs=25,
                batch_size=128,
                shuffle=True,
                validation_data=(x_test, x_test))

In [ ]:
y_test = autoencoder.predict(x_test[0:5])
plotn(5,x_test)
plotn(5,y_test)

In [ ]:
encoder = Model(input_img, encoded)
encoded_imgs = encoder.predict(x_test[0:5])

In [ ]:
plotn(5,encoded_imgs.reshape(5,-1,8))

In [ ]:
print(encoded_imgs.max(),encoded_imgs.min())
res = decoder.predict(7*np.random.rand(7,4,4,8))
plotn(7,res)

> **Task 1**: Autoencoder ko bahut chhote latent vector size, jaise 2, ke saath train karne ki koshish karein, aur alag-alag digits ke corresponding points ko plot karein. *Hint: Convolutional part ke baad vector ko desired size tak kam karne ke liye fully connected Dense layer use karein.*

> **Task 2**: Alag-alag digits se shuru karke, unki latent space representation nikaalein, aur dekhein ki latent space mein thoda noise add karne se resulting digits par kya asar padta hai.

## Noise hatana

Autoencoders ko images se noise hatane ke liye effectively use kiya ja sakta hai. Denoiser ko train karne ke liye, hum noise-free images se shuru karenge aur unmein artificial noise add karenge. Phir, hum autoencoder ko input ke roop mein noisy images aur output ke roop mein noise-free images denge.

Chalo dekhte hain yeh MNIST ke liye kaise kaam karta hai:

In [ ]:
def noisify(data):
  return np.clip(data+np.random.normal(loc=0.5,scale=0.5,size=data.shape),0.,1.)

x_train_noise = noisify(x_train)
x_test_noise = noisify(x_test)

plotn(5,x_train_noise)

In [ ]:
autoencoder.fit(x_train_noise, x_train,
                epochs=25,
                batch_size=128,
                shuffle=True,
                validation_data=(x_test_noise, x_test))

In [ ]:
y_test = autoencoder.predict(x_test_noise[0:5])
plotn(5,x_test_noise)
plotn(5,y_test)

> **Exercise:** Dekhein ki MNIST digits par train kiya gaya denoiser alag-alag images ke liye kaise kaam karta hai. Jaise, aap [Fashion MNIST](https://keras.io/api/datasets/fashion_mnist/) dataset le sakte hain, jiska image size same hai. Dhyan rakhiye ki denoiser sirf usi type ki images par achhe se kaam karta hai jis par use train kiya gaya tha (yaani input data ki same probability distribution ke liye).

## Super-resolution

Denoiser ki tarah, hum autoencoders ko image ka resolution badhane ke liye train kar sakte hain. Super-resolution network ko train karne ke liye, hum high-resolution images se shuru karenge aur unhein automatically downscale karenge taaki network input ban sake. Uske baad, hum autoencoder ko chhote size ki images input aur high-resolution images output ke roop mein denge.

Chalo MNIST ko 14x14 mein downscale karte hain:

In [ ]:
x_train_lr = tf.keras.layers.AveragePooling2D()(x_train).numpy()
x_test_lr = tf.keras.layers.AveragePooling2D()(x_test).numpy()
plotn(5,x_train_lr)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, UpSampling2D, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.losses import binary_crossentropy,mse

input_img = Input(shape=(14, 14, 1))

x = Conv2D(16, (3, 3), activation='relu', padding='same')(input_img)
x = MaxPooling2D((2, 2), padding='same')(x)
x = Conv2D(8, (3, 3), activation='relu', padding='same')(x)
encoded = MaxPooling2D((2, 2), padding='same')(x)

encoder = Model(input_img,encoded)

input_rep = Input(shape=(4,4,8))

x = Conv2D(8, (3, 3), activation='relu', padding='same')(input_rep)
x = UpSampling2D((2, 2))(x)
x = Conv2D(8, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)
x = Conv2D(16, (3, 3), activation='relu')(x)
x = UpSampling2D((2, 2))(x)
decoded = Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

decoder = Model(input_rep,decoded)

autoencoder = Model(input_img, decoder(encoder(input_img)))
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

In [ ]:
autoencoder.fit(x_train_lr, x_train,
                epochs=25,
                batch_size=128,
                shuffle=True,
                validation_data=(x_test_lr, x_test))

In [ ]:
y_test_lr = autoencoder.predict(x_test_lr[0:5])
plotn(5,x_test_lr)
plotn(5,y_test_lr)

> **Exercise**: [CIFAR-10](https://keras.io/api/datasets/cifar10/) par 2x aur 4x upscaling ke liye super-resolution networks ko train karne ki koshish karein. 4x upscaling model ke liye input mein noise use karein aur result dekhein.

## Variational Auto-Encoders (VAE)

Traditional auto-encoders input data ki dimensionality ko kisi tarah kam karte hain, aur input images ke important features pehchaante hain. Lekin, latent vectors aksar zyada samajh mein nahi aate. Dusre shabdon mein, agar hum MNIST dataset ka example lein, toh yeh samajhna aasaan nahi hai ki kaun se digits alag-alag latent vectors se match karte hain, kyunki paas-paas ke latent vectors zaroori nahi ki same digit ko represent karein.

Doosri taraf, *generative* models ko train karne ke liye latent space ki kuch samajh hona behtar hota hai. Yahi idea humein **Variational Auto-Encoder** (VAE) ki taraf le jaata hai.

VAE ek aisa auto-encoder hai jo latent parameters ka *statistical distribution* seekhta hai, jise **latent distribution** kehte hain. Jaise, hum maan sakte hain ki latent vectors $N(\mathrm{z\_mean},e^{\mathrm{z\_log\_sigma}})$ ke roop mein distributed honge, jahan $\mathrm{z\_mean}, \mathrm{z\_log\_sigma} \in\mathbb{R}^d$. VAE mein encoder in parameters ko predict karna seekhta hai, aur phir decoder is distribution se ek random vector lekar object ko reconstruct karta hai.

Summary mein:

 * Input vector se, hum `z_mean` aur `z_log_sigma` estimate karte hain (standard deviation ko directly predict karne ke bajaye, hum uska logarithm predict karte hain)
 * Hum distribution $N(\mathrm{z\_mean},e^{\mathrm{z\_log\_sigma}})$ se ek vector `sample` sample karte hain
 * Decoder `sample` ko input vector ki tarah leke original image ko decode karne ki koshish karta hai

 <img src="images/vae.png" width="50%">

In [ ]:
intermediate_dim = 512
latent_dim = 2
batch_size = 128

tf.compat.v1.disable_eager_execution()

inputs = Input(shape=(784,))
h = Dense(intermediate_dim, activation='relu')(inputs)
z_mean = Dense(latent_dim)(h)
z_log_sigma = Dense(latent_dim)(h)

In [ ]:
@tf.function
def sampling(args):
    z_mean, z_log_sigma = args
    bs = tf.shape(z_mean)[0]
    epsilon = tf.random.normal(shape=(bs, latent_dim))
    return z_mean + tf.exp(z_log_sigma) * epsilon

z = Lambda(sampling)([z_mean, z_log_sigma])

In [ ]:
encoder = Model(inputs, [z_mean, z_log_sigma, z])

latent_inputs = Input(shape=(latent_dim,))
x = Dense(intermediate_dim, activation='relu')(latent_inputs)
outputs = Dense(784, activation='sigmoid')(x)

decoder = Model(latent_inputs, outputs)

outputs = decoder(encoder(inputs)[2])

vae = Model(inputs, outputs)

Variational auto-encoders ek complex loss function use karte hain, jo do parts mein divide hota hai:
* **Reconstruction loss** woh loss function hai, jo dikhata hai ki reconstructed image target ke kitni kareeb hai (yeh MSE ho sakta hai). Yeh wahi loss function hai jo normal auto-encoders mein use hota hai.
* **KL loss**, jo yeh ensure karta hai ki latent variable ka distribution normal distribution ke kareeb bana rahe. Yeh [Kullback-Leibler divergence](https://www.countbayesie.com/blog/2017/5/9/kullback-leibler-divergence-explained) ke concept par based hai - yeh ek metric hai jo estimate karne ke liye use hota hai ki do statistical distributions kitne similar hain.

In [ ]:
@tf.function
def vae_loss(x1,x2):
  reconstruction_loss = mse(x1,x2)*784
  tmp = 1 + z_log_sigma - tf.square(z_mean) - tf.exp(z_log_sigma)
  kl_loss = -0.5*tf.reduce_sum(tmp, axis=-1)
  return tf.convert_to_tensor(tf.reduce_mean(reconstruction_loss + kl_loss))

vae.compile(optimizer='rmsprop', loss=vae_loss)

In [ ]:
x_train_flat = x_train.reshape((len(x_train), np.prod(x_train.shape[1:])))
x_test_flat = x_test.reshape((len(x_test), np.prod(x_test.shape[1:])))

vae.fit(x_train_flat, x_train_flat,
        shuffle=True,
        epochs=25,
        batch_size=batch_size,
        validation_data=(x_test_flat, x_test_flat))

In [ ]:
y_test = vae.predict(x_test_flat[0:5])
plotn(5,x_test_flat)
plotn(5,y_test)

In [ ]:
x_test_encoded = encoder.predict(x_test_flat)[0]
plt.figure(figsize=(6, 6))
plt.scatter(x_test_encoded[:, 0], x_test_encoded[:, 1], c=y_testclass)
plt.colorbar()
plt.show()

In [ ]:
def plotsample(n):
  dx = np.linspace(-1,1,n)
  dy = np.linspace(-1,1,n)
  fig,ax = plt.subplots(n,n)
  for i,xi in enumerate(dx):
    for j,xj in enumerate(dy):
      res = decoder.predict(np.array([xi,xj]).reshape(-1,2))[0]
      ax[i,j].imshow(res.reshape(28,28))
      ax[i,j].axis('off')
  plt.show()
  
plotsample(10)

> **Task**: Hamare sample mein humne fully connected VAE train kiya hai. Ab traditional auto-encoder se CNN lein aur CNN-based VAE banayein.

## Additional material

* [NeuroHive par blog post](https://neurohive.io/ru/osnovy-data-science/variacionnyj-avtojenkoder-vae/)
* [Variational Autoencoders ki explanation](https://kvfrans.com/variational-autoencoders-explained/)

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->